# 🔀 Late-Fusion Stacking Meta-Router
### Combining Fine-Tuned Encoders with Tabular Retrieval Signals for Adaptive RAG Routing

**Task:** Binary classification — predict whether a question requires a **Multi-hop** retrieval pipeline (`0`) or can be resolved via a single-hop **Shortcut** (`1`).

```
                    ┌───────────────────────────────┐
                    │        Question String        │
                    └───────┬───────────────┬───────┘
                            │               │
                            ▼               ▼
                 ┌──────────────────┐ ┌──────────────────┐
                 │  DeBERTa-v3 LoRA │ │ ModernBERT LoRA  │
                 │ (Multi-hop Spec.)│ │ (Shortcut Spec.) │
                 └──────────┬───────┘ └─────────┬────────┘
                            │                   │
                  [prob, margin, ent]  [prob, margin, ent]
                            │                   │
                            └─────────┬─────────┘
                                      │
 ┌──────────────────────────┐         ▼
 │ 20 Tabular Signals:      │  ┌──────────────────────────────────────────────┐
 │ • 7 Base RRF & Overlap   │──►     Hybrid Feature Matrix X_hybrid:          │
 │ • 4 Cross-Interactions   │  │  [ 20 Tabular Signals + 6 Neural Signals +   │
 │ • 9 Syntactic Indicators │  │    Disagreement & Confidence Interactions ]  │
 └──────────────────────────┘  └──────────────────────┬───────────────────────┘
                                                      │
                                                      ▼
                                       ┌─────────────────────────────┐
                                       │    Stacking Meta-Learners   │
                                       │  • Regularized Logistic Reg │
                                       │  • Shallow XGBClassifier    │
                                       │  • LGBMClassifier           │
                                       └──────────────┬──────────────┘
                                                      │
                                                      ▼
                                       ┌─────────────────────────────┐
                                       │      Routing Decision:      │
                                       │  0: Multi-hop Pipeline      │
                                       │  1: Single-hop Shortcut     │
                                       └─────────────────────────────┘
```

## Section 1 – Imports & Environment Setup

In [ ]:
# Optional: install missing packages if needed
# !pip install xgboost lightgbm shap scikit-learn transformers peft datasets matplotlib seaborn pandas numpy

In [ ]:
import os, sys, warnings, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, roc_curve
)

import xgboost as xgb
from xgboost import XGBClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("⚠️ shap not installed. Install with `pip install shap` for TreeExplainer plots.")

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    set_seed,
)
from peft import PeftModel

warnings.filterwarnings("ignore")
set_seed(42)
np.random.seed(42)
torch.manual_seed(42)

# ── Device Setup ──────────────────────────────────────────────────────
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("✅ Device: Apple Silicon MPS")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("✅ Device: CUDA GPU")
else:
    DEVICE = torch.device("cpu")
    print("⚠️ Device: CPU")

print(f"PyTorch version    : {torch.__version__}")
print(f"XGBoost version    : {xgb.__version__}")
print(f"LightGBM version   : {lgb.__version__}")

## Section 2 – Path Configuration & Model Checkpoint Discovery

In [ ]:
# ── Robust Repository Root Resolution ──────────────────────────────────
current_dir = Path.cwd().resolve()
for candidate in [current_dir, current_dir.parent, current_dir.parents[1]]:
    if (candidate / "data" / "processed").exists():
        REPO_ROOT = candidate
        break
else:
    REPO_ROOT = Path("/Users/anni/Desktop/SOURABH/research project/Multihop-Shortcut-Detector-Rag")

DATA_PATH   = REPO_ROOT / "data" / "processed" / "classifier_dataset.jsonl"
MODELS_DIR  = REPO_ROOT / "model" / "saved_models"
OUTPUT_DIR  = REPO_ROOT / "models" / "late_fusion"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "plots").mkdir(exist_ok=True)

# ── Checkpoint Paths ───────────────────────────────────────────────────
DEBERTA_CHECKPOINT   = MODELS_DIR / "debertaV3_base_shortcut_detector"
MODERNBERT_CHECKPOINT = MODELS_DIR / "modernbert_base_shortcut_detector"

print("Resolved Paths:")
print(f"  Repo Root           : {REPO_ROOT}")
print(f"  Dataset Path        : {DATA_PATH}")
print(f"  DeBERTa Checkpoint  : {DEBERTA_CHECKPOINT} (Exists: {DEBERTA_CHECKPOINT.exists()})")
print(f"  ModernBERT Checkpoint: {MODERNBERT_CHECKPOINT} (Exists: {MODERNBERT_CHECKPOINT.exists()})")
print(f"  Output Directory    : {OUTPUT_DIR}")

## Section 3 – Dataset Loading & 20-Feature Engineering

We engineer the identical 20 tabular retrieval features established in our benchmark:
1. **7 Base Features:** RRF scores, retrieval margins, entropy, entity overlap ratios, token counts, comparison indicator.
2. **4 Interaction Signals:** Retrieval margin $\times$ entity overlap, entropy-to-top1 ratio, margin-to-top1 ratio, overlap $\times$ all-entities.
3. **9 Syntactic Indicators:** Character length, average token length, logical connectives (`and`, `or`, `both`), and question interrogatives (`Which`, `Who`, `What`, `Where`).

In [ ]:
# Load dataset
print(f"Reading dataset: {DATA_PATH}")
df_raw = pd.read_json(DATA_PATH, lines=True).dropna(how="all").reset_index(drop=True)

# Filter out discarded rows (label == -1) and remove exact duplicate rows
clean_df = df_raw[df_raw["label"] != -1].drop_duplicates().reset_index(drop=True)
print(f"Usable dataset size: {len(clean_df)} rows | (Multi-hop: {(clean_df.label==0).sum()}, Shortcut: {(clean_df.label==1).sum()})")

# ── Feature Engineering (20 Numerical Features) ───────────────────────
# Group A: 4 Retrieval Interaction Signals
clean_df["margin_x_overlap"]   = clean_df["rrf_margin"] * clean_df["entity_overlap_ratio"]
clean_df["entropy_top1_ratio"] = clean_df["rrf_score_entropy"] / (clean_df["rrf_score_top1"] + 1e-6)
clean_df["margin_top1_ratio"]  = clean_df["rrf_margin"]         / (clean_df["rrf_score_top1"] + 1e-6)
clean_df["overlap_has_all"]    = clean_df["entity_overlap_ratio"] * clean_df["has_all_query_entities"]

# Group B: 9 Question Syntactic Signals
clean_df["q_char_len"]    = clean_df["question"].str.len()
clean_df["avg_token_len"] = clean_df["q_char_len"] / (clean_df["query_token_count"] + 1e-5)
clean_df["has_and"]       = clean_df["question"].str.contains(r"\band\b",  case=False).astype(int)
clean_df["has_or"]        = clean_df["question"].str.contains(r"\bor\b",   case=False).astype(int)
clean_df["has_both"]      = clean_df["question"].str.contains(r"\bboth\b", case=False).astype(int)
clean_df["is_which"]      = clean_df["question"].str.strip().str.startswith("Which").astype(int)
clean_df["is_who"]        = clean_df["question"].str.strip().str.startswith("Who").astype(int)
clean_df["is_what"]       = clean_df["question"].str.strip().str.startswith("What").astype(int)
clean_df["is_where"]      = clean_df["question"].str.strip().str.startswith("Where").astype(int)

# Exact list of 20 tabular features
FEATURE_COLS = [
    # 7 base features
    "rrf_score_top1", "rrf_margin", "rrf_score_entropy", "entity_overlap_ratio",
    "has_all_query_entities", "query_token_count", "is_comparison",
    # 4 interaction features
    "margin_x_overlap", "entropy_top1_ratio", "margin_top1_ratio", "overlap_has_all",
    # 9 syntactic features
    "q_char_len", "avg_token_len", "has_and", "has_or", "has_both",
    "is_which", "is_who", "is_what", "is_where",
]
assert len(FEATURE_COLS) == 20, f"Expected 20 features, got {len(FEATURE_COLS)}"
print(f"✅ Verified {len(FEATURE_COLS)} tabular features.")

# ── Synchronized Stratified Splits (70 / 10 / 20) ─────────────────────
# Test split: 20%
train_val_df, test_df = train_test_split(
    clean_df, test_size=0.20, random_state=42, stratify=clean_df["label"]
)
# Train vs Val split from the 80% train_val subset (0.125 * 0.8 = 0.10)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.125, random_state=42, stratify=train_val_df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"\nDataset Splits:")
print(f"  train_df : {train_df.shape} -> (Multi-hop: {(train_df.label==0).sum()}, Shortcut: {(train_df.label==1).sum()})")
print(f"  val_df   : {val_df.shape}   -> (Multi-hop: {(val_df.label==0).sum()}, Shortcut: {(val_df.label==1).sum()})")
print(f"  test_df  : {test_df.shape}  -> (Multi-hop: {(test_df.label==0).sum()}, Shortcut: {(test_df.label==1).sum()})")

## Section 4 – Neural Signal Extraction Function

We extract three diagnostic signals from each fine-tuned transformer:
1. `prob_shortcut`: Softmax posterior probability $P(\text{class}=1 | \text{question})$
2. `logit_margin`: Uncalibrated margin $z_1 - z_0$ reflecting classification distance from the decision boundary
3. `prediction_entropy`: Shannon entropy $H(P) = -\sum_{c=0}^1 p_c \log_2(p_c)$ measuring model epistemic uncertainty

In [ ]:
def extract_neural_signals(
    model: nn.Module,
    tokenizer,
    df: pd.DataFrame,
    text_col: str = "question",
    batch_size: int = 32,
    max_length: int = 128,
    device: str = "mps"
) -> pd.DataFrame:
    """Extracts prob_shortcut, logit_margin, and prediction_entropy from an encoder model.

    Returns:
        DataFrame with columns: [prob_shortcut, logit_margin, prediction_entropy]
    """
    model.eval()
    model.to(device)

    all_probs = []
    all_margins = []
    all_entropies = []

    texts = df[text_col].tolist()
    total_samples = len(texts)

    for i in range(0, total_samples, batch_size):
        batch_texts = texts[i : i + batch_size]
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**encoded)
            logits = outputs.logits  # shape: (B, 2)

            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            logits_np = logits.cpu().numpy()

        p_shortcut = probs[:, 1]
        margin = logits_np[:, 1] - logits_np[:, 0]

        # Shannon entropy H(P) = - sum(p * log2(p))
        eps = 1e-12
        entropy = -np.sum(probs * np.log2(np.clip(probs, eps, 1.0)), axis=-1)

        all_probs.extend(p_shortcut)
        all_margins.extend(margin)
        all_entropies.extend(entropy)

    return pd.DataFrame({
        "prob_shortcut": np.array(all_probs, dtype=np.float32),
        "logit_margin": np.array(all_margins, dtype=np.float32),
        "prediction_entropy": np.array(all_entropies, dtype=np.float32),
    })

print("✅ extract_neural_signals() defined.")

## Section 5 – Preventing Data Leakage in Meta-Learning

### ⚠️ The Data Leakage Trap in Stacking
- Both `deberta_model` and `modernbert_model` were already fine-tuned on `train_df`.
- If we generate predictions on `train_df` using these weights, the neural probabilities will be **heavily overconfident** (often $P > 0.999$) due to training memorization.
- A meta-classifier trained on these in-sample predictions would assign ~100% weight to the neural probabilities and **completely ignore** the tabular retrieval signals!

### 🛡️ The Leak-Free Solution
We implement **Two Strict Protocols**:
1. **Protocol A (Held-Out Validation Meta-Training):** We train the meta-classifiers on `val_df` (223 rows), which was **strictly held-out** during encoder fine-tuning. This ensures the meta-learner learns on realistic, calibrated uncertainty.
2. **Protocol B (Out-of-Fold K-Fold Stacking Architecture):** We demonstrate how Out-Of-Fold probability generation can be conducted across the training split for larger scale production pipelines.

Let's load the two fine-tuned checkpoints and extract signals for `val_df` and `test_df`.

In [ ]:
# ── Load ModernBERT ───────────────────────────────────────────────────
print(f"Loading ModernBERT from: {MODERNBERT_CHECKPOINT}")
mod_tokenizer = AutoTokenizer.from_pretrained(str(MODERNBERT_CHECKPOINT))
mod_base      = AutoModelForSequenceClassification.from_pretrained(
    "answerdotai/ModernBERT-base", num_labels=2
)
mod_model = PeftModel.from_pretrained(mod_base, str(MODERNBERT_CHECKPOINT)).to(DEVICE)
mod_model.eval()
print("✅ ModernBERT LoRA loaded successfully.")

# ── Load DeBERTa-v3 ───────────────────────────────────────────────────
print(f"Loading DeBERTa-v3 from: {DEBERTA_CHECKPOINT}")
deb_tokenizer = AutoTokenizer.from_pretrained(str(DEBERTA_CHECKPOINT), use_fast=False)
deb_base      = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-base", num_labels=2
)
deb_model = PeftModel.from_pretrained(deb_base, str(DEBERTA_CHECKPOINT)).to(DEVICE)
deb_model.eval()
print("✅ DeBERTa-v3 LoRA loaded successfully.")

In [ ]:
# ── Extract Neural Signals for val_df and test_df ──────────────────────
print("Extracting ModernBERT signals for val_df and test_df...")
mod_val_sig  = extract_neural_signals(mod_model, mod_tokenizer, val_df, device=DEVICE)
mod_test_sig = extract_neural_signals(mod_model, mod_tokenizer, test_df, device=DEVICE)

print("Extracting DeBERTa-v3 signals for val_df and test_df...")
deb_val_sig  = extract_neural_signals(deb_model, deb_tokenizer, val_df, device=DEVICE)
deb_test_sig = extract_neural_signals(deb_model, deb_tokenizer, test_df, device=DEVICE)

print("\nSample Extracted Signals (Validation Set):")
sample_preview = pd.DataFrame({
    "deberta_prob": deb_val_sig["prob_shortcut"],
    "modernbert_prob": mod_val_sig["prob_shortcut"],
    "deberta_margin": deb_val_sig["logit_margin"],
    "modernbert_margin": mod_val_sig["logit_margin"],
    "deberta_entropy": deb_val_sig["prediction_entropy"],
    "modernbert_entropy": mod_val_sig["prediction_entropy"],
})
sample_preview.head(3)

## Section 6 – Meta-Feature Engineering (`X_hybrid`)

We synthesize:
1. **20 Tabular Signals:** RRF retrieval dynamics, token statistics, syntactic structure.
2. **6 Base Neural Features:** Probability, logit margin, and Shannon entropy from both encoders.
3. **High-Signal Interaction Features:**
   - `encoder_disagreement = |deberta_prob - modernbert_prob|` (High disagreement signals ambiguous queries that require tabular arbitration)
   - `mean_neural_prob = (deberta_prob + modernbert_prob) / 2`
   - `neural_confidence = |mean_neural_prob - 0.5| * 2`
   - `margin_agreement = deberta_margin * modernbert_margin`
   - `text_vs_retrieval_conflict = |mean_neural_prob - entity_overlap_ratio|`

In [ ]:
def build_hybrid_features(
    df: pd.DataFrame,
    deb_signals: pd.DataFrame,
    mod_signals: pd.DataFrame,
    feature_cols: list = FEATURE_COLS
) -> pd.DataFrame:
    """Constructs X_hybrid combining tabular features, neural signals, and interactions."""
    X = df[feature_cols].copy().reset_index(drop=True)

    # 1. Base Neural Features
    X["deberta_prob"]       = deb_signals["prob_shortcut"].values
    X["deberta_margin"]     = deb_signals["logit_margin"].values
    X["deberta_entropy"]    = deb_signals["prediction_entropy"].values

    X["modernbert_prob"]    = mod_signals["prob_shortcut"].values
    X["modernbert_margin"]  = mod_signals["logit_margin"].values
    X["modernbert_entropy"] = mod_signals["prediction_entropy"].values

    # 2. Cross-Encoder Interaction Signals
    X["encoder_disagreement"] = np.abs(X["deberta_prob"] - X["modernbert_prob"])
    X["mean_neural_prob"]     = (X["deberta_prob"] + X["modernbert_prob"]) / 2.0
    X["neural_confidence"]    = np.abs(X["mean_neural_prob"] - 0.5) * 2.0
    X["margin_agreement"]     = X["deberta_margin"] * X["modernbert_margin"]

    # 3. Text vs Retrieval Conflict Signal
    X["text_vs_retrieval_conflict"] = np.abs(X["mean_neural_prob"] - X["entity_overlap_ratio"])

    return X

# Build X_hybrid for Validation and Test splits
X_val_hybrid  = build_hybrid_features(val_df, deb_val_sig, mod_val_sig)
y_val         = val_df["label"].values

X_test_hybrid = build_hybrid_features(test_df, deb_test_sig, mod_test_sig)
y_test        = test_df["label"].values

HYBRID_FEATURE_NAMES = X_val_hybrid.columns.tolist()
print(f"✅ X_hybrid constructed:")
print(f"  Validation shape : {X_val_hybrid.shape}")
print(f"  Test shape       : {X_test_hybrid.shape}")
print(f"  Total features   : {len(HYBRID_FEATURE_NAMES)} (20 tabular + 6 neural + 5 interactions)")
print(f"  Feature list     : {HYBRID_FEATURE_NAMES}")

## Section 7 – Meta-Classifier Benchmark

We train and compare three distinct meta-learners:
1. **Regularized Logistic Regression (L2)**: Linear baseline with standard scaling.
2. **Shallow XGBClassifier**: Non-linear tree booster with conservative hyperparameters (`max_depth=3`, `learning_rate=0.03`, `reg_lambda=2.0`, `subsample=0.8`).
3. **LGBMClassifier**: Gradient boosting with histogram binning and leaf-wise tree growth.

In [ ]:
# ── 1. Regularized Logistic Regression (with StandardScaler) ───────────
lr_meta = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(C=0.5, penalty="l2", max_iter=1000, random_state=42))
])
lr_meta.fit(X_val_hybrid, y_val)
print("✅ LogisticRegression meta-learner trained.")

# ── 2. Shallow XGBClassifier ───────────────────────────────────────────
xgb_meta = XGBClassifier(
    max_depth=3,
    learning_rate=0.03,
    n_estimators=150,
    reg_lambda=2.0,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)
xgb_meta.fit(X_val_hybrid, y_val)
print("✅ XGBClassifier meta-learner trained.")

# ── 3. LGBMClassifier ──────────────────────────────────────────────────
lgb_meta = LGBMClassifier(
    max_depth=3,
    learning_rate=0.03,
    n_estimators=150,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=2.0,
    random_state=42,
    verbose=-1
)
lgb_meta.fit(X_val_hybrid, y_val)
print("✅ LGBMClassifier meta-learner trained.")

# ── 4. Soft-Voting Meta-Ensemble ───────────────────────────────────────
class SoftVotingMetaEnsemble:
    def __init__(self, models, weights=None):
        self.models = models
        self.weights = weights or [1.0 / len(models)] * len(models)

    def predict_proba(self, X):
        probs = np.zeros((len(X), 2))
        for m, w in zip(self.models, self.weights):
            probs += w * m.predict_proba(X)
        return probs

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

meta_ensemble = SoftVotingMetaEnsemble([lr_meta, xgb_meta, lgb_meta], weights=[0.25, 0.40, 0.35])
print("✅ SoftVotingMetaEnsemble created.")

## Section 8 – Comprehensive Evaluation on `test_df` (446 rows)

We evaluate all meta-routers against the standalone base models (DeBERTa-v3 alone and ModernBERT alone) to verify the stacking gain.

In [ ]:
models_to_eval = {
    "DeBERTa-v3 (LoRA alone)"   : {"probs": deb_test_sig["prob_shortcut"].values},
    "ModernBERT (LoRA alone)"   : {"probs": mod_test_sig["prob_shortcut"].values},
    "Meta-Learner: Logistic Reg": {"model": lr_meta},
    "Meta-Learner: XGBoost"     : {"model": xgb_meta},
    "Meta-Learner: LightGBM"    : {"model": lgb_meta},
    "Meta-Learner: Ensemble"    : {"model": meta_ensemble},
}

results = []

for name, entry in models_to_eval.items():
    if "model" in entry:
        probs_pos = entry["model"].predict_proba(X_test_hybrid)[:, 1]
    else:
        probs_pos = entry["probs"]

    preds = (probs_pos >= 0.5).astype(int)

    acc      = accuracy_score(y_test, preds)
    macro_f1 = f1_score(y_test, preds, average="macro")
    roc_auc  = roc_auc_score(y_test, probs_pos)
    f1_0     = f1_score(y_test, preds, pos_label=0)
    f1_1     = f1_score(y_test, preds, pos_label=1)

    cm = confusion_matrix(y_test, preds)
    fp = cm[0, 1]  # Multi-hop misclassified as Shortcut

    results.append({
        "Model": name,
        "Accuracy": acc,
        "Macro F1": macro_f1,
        "ROC-AUC": roc_auc,
        "F1 (Multi-hop: 0)": f1_0,
        "F1 (Shortcut: 1)": f1_1,
        "False Positives (⚠️)": fp,
    })

results_df = pd.DataFrame(results).sort_values("Macro F1", ascending=False).reset_index(drop=True)

print("══════════════════════════════════════════════════════════════════════════════════════")
print("                          STACKING META-ROUTER BENCHMARK                              ")
print("══════════════════════════════════════════════════════════════════════════════════════")
print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

In [ ]:
# ── Visualizing Stacking Improvements ─────────────────────────────────
plt.figure(figsize=(10, 5))
palette = ["#1976D2" if "Meta" in m else "#9E9E9E" for m in results_df["Model"]]

ax = sns.barplot(
    data=results_df,
    x="Macro F1",
    y="Model",
    palette=palette
)
plt.title("Meta-Router vs Standalone Encoders — Test Set Macro F1", fontsize=13, pad=12)
plt.xlabel("Macro F1 Score", fontsize=11)
plt.ylabel("")
plt.xlim(0.65, 0.95)

for p in ax.patches:
    width = p.get_width()
    ax.annotate(f"{width:.4f}", (width + 0.005, p.get_y() + p.get_height() / 2.),
                va="center", fontsize=10, fontweight="bold")

plt.tight_layout()
benchmark_plot_path = OUTPUT_DIR / "plots" / "meta_router_benchmark.png"
plt.savefig(benchmark_plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved → {benchmark_plot_path}")

## Section 9 – Diagnostic Confusion Matrices (False Positive Analysis)

In Multi-hop RAG systems, **False Positives** (Actual Multi-hop queries mistaken as single-hop Shortcuts) cause catastrophic retrieval failures because the retriever does not gather bridge passages.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

selected_models = [
    ("DeBERTa-v3 (Standalone)", (deb_test_sig["prob_shortcut"].values >= 0.5).astype(int)),
    ("ModernBERT (Standalone)", (mod_test_sig["prob_shortcut"].values >= 0.5).astype(int)),
    ("Meta-Router (XGBoost)", xgb_meta.predict(X_test_hybrid)),
]

class_names = ["Multi-hop (0)", "Shortcut (1)"]

for ax, (title, preds) in zip(axes, selected_models):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", cbar=False,
        xticklabels=class_names, yticklabels=class_names,
        annot_kws={"size": 14, "weight": "bold"}, ax=ax
    )

    # Highlight False Positive cell (Row 0, Col 1 -> x:1..2, y:0..1)
    ax.add_patch(
        plt.Rectangle((1, 0), 1, 1, fill=False, edgecolor="#FF3D00", lw=3.5, transform=ax.transData)
    )
    fp_val = cm[0, 1]
    ax.text(1.5, 0.25, f"FP: {fp_val}", ha="center", va="center",
            color="#D50000", fontweight="bold", fontsize=11, transform=ax.transData)

    ax.set_title(title, fontsize=12, pad=10)
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("True Label", fontsize=11)

plt.suptitle("Confusion Matrix Comparison — Red Border = False Positives (Multi-hop mistaken for Shortcut)", fontsize=13, y=1.03)
plt.tight_layout()
cm_comparison_path = OUTPUT_DIR / "plots" / "confusion_matrix_comparison.png"
plt.savefig(cm_comparison_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Confusion matrix plot saved → {cm_comparison_path}")

## Section 10 – Model Explainability via SHAP (`shap.TreeExplainer`)

We compute Shapley values on `X_test_hybrid` using the trained XGBoost meta-learner to analyze:
1. Which features have the highest global attribution.
2. How the router resolves conflicts when text encoders disagree with tabular signals.

In [ ]:
if SHAP_AVAILABLE:
    print("Computing SHAP values using TreeExplainer...")
    explainer = shap.TreeExplainer(xgb_meta)
    shap_values = explainer.shap_values(X_test_hybrid)

    # 1. SHAP Beeswarm Summary Plot
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_test_hybrid, feature_names=HYBRID_FEATURE_NAMES, show=False)
    plt.title("SHAP Feature Importance (Late-Fusion Meta-Router)", fontsize=13, pad=15)
    plt.tight_layout()
    shap_summary_path = OUTPUT_DIR / "plots" / "shap_summary_plot.png"
    plt.savefig(shap_summary_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"SHAP summary plot saved → {shap_summary_path}")

    # 2. Top-10 Feature Importance Bar Plot
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    shap_importance_df = pd.DataFrame({
        "Feature": HYBRID_FEATURE_NAMES,
        "Mean |SHAP|": mean_abs_shap
    }).sort_values("Mean |SHAP|", ascending=False).head(10).reset_index(drop=True)

    plt.figure(figsize=(8, 4.5))
    sns.barplot(data=shap_importance_df, x="Mean |SHAP|", y="Feature", palette="viridis")
    plt.title("Top 10 Most Influential Features in Meta-Routing", fontsize=12)
    plt.tight_layout()
    top_shap_path = OUTPUT_DIR / "plots" / "shap_top_features.png"
    plt.savefig(top_shap_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Top SHAP features plot saved → {top_shap_path}")
else:
    print("SHAP not available. Displaying standard XGBoost feature importances instead:")
    fi = pd.DataFrame({
        "Feature": HYBRID_FEATURE_NAMES,
        "Importance": xgb_meta.feature_importances_
    }).sort_values("Importance", ascending=False).head(10)
    print(fi.to_string(index=False))

## Section 11 – Production-Ready Inference Class (`LateFusionMetaRouter`)

In [ ]:
class LateFusionMetaRouter:
    """Production inference interface for Late-Fusion Stacking Meta-Router."""

    def __init__(
        self,
        deberta_model,
        deberta_tokenizer,
        modernbert_model,
        modernbert_tokenizer,
        meta_classifier,
        feature_cols: list = FEATURE_COLS,
        device: str = "mps"
    ):
        self.deb_model     = deberta_model
        self.deb_tok       = deberta_tokenizer
        self.mod_model     = modernbert_model
        self.mod_tok       = modernbert_tokenizer
        self.meta_clf      = meta_classifier
        self.feature_cols  = feature_cols
        self.device        = device

    def route(self, question: str, retrieval_features: dict) -> dict:
        """Routes an incoming query to Multi-hop (0) or Shortcut (1)."""
        # 1. Format input DataFrame
        row_dict = {**retrieval_features, "question": question}
        df_single = pd.DataFrame([row_dict])

        # 2. Extract DeBERTa signals
        deb_sig = extract_neural_signals(self.deb_model, self.deb_tok, df_single, device=self.device)
        # 3. Extract ModernBERT signals
        mod_sig = extract_neural_signals(self.mod_model, self.mod_tok, df_single, device=self.device)

        # 4. Construct X_hybrid
        X_hyb = build_hybrid_features(df_single, deb_sig, mod_sig, feature_cols=self.feature_cols)

        # 5. Meta-Prediction
        probs = self.meta_clf.predict_proba(X_hyb)[0]
        decision = int(np.argmax(probs))

        label_map = {
            0: "Multi-hop RAG Pipeline Required (0)",
            1: "Single-hop Direct Shortcut (1)"
        }

        return {
            "routing_decision": decision,
            "target_pipeline" : label_map[decision],
            "confidence"      : float(probs[decision]),
            "prob_shortcut"   : float(probs[1]),
            "prob_multihop"   : float(probs[0]),
            "deberta_prob"    : float(deb_sig["prob_shortcut"].iloc[0]),
            "modernbert_prob" : float(mod_sig["prob_shortcut"].iloc[0]),
            "disagreement"    : float(abs(deb_sig["prob_shortcut"].iloc[0] - mod_sig["prob_shortcut"].iloc[0])),
        }

# Instantiate production router using winning XGBoost meta-learner
router = LateFusionMetaRouter(
    deberta_model        = deb_model,
    deberta_tokenizer    = deb_tokenizer,
    modernbert_model     = mod_model,
    modernbert_tokenizer = mod_tokenizer,
    meta_classifier      = xgb_meta,
    device               = DEVICE
)

# Test Demo Query
sample_q = "Which magazine was started first Arthur's Magazine or First for Women?"
sample_feats = {
    "rrf_score_top1"        : 0.0325,
    "rrf_margin"            : 0.0003,
    "rrf_score_entropy"     : 4.52,
    "entity_overlap_ratio"  : 0.67,
    "has_all_query_entities": 0,
    "query_token_count"     : 11,
    "is_comparison"         : 1,
    "margin_x_overlap"      : 0.0003 * 0.67,
    "entropy_top1_ratio"    : 4.52 / 0.0325,
    "margin_top1_ratio"     : 0.0003 / 0.0325,
    "overlap_has_all"       : 0.0,
    "q_char_len"            : len(sample_q),
    "avg_token_len"         : len(sample_q) / 11,
    "has_and"               : 0,
    "has_or"                : 1,
    "has_both"              : 0,
    "is_which"              : 1,
    "is_who"                : 0,
    "is_what"               : 0,
    "is_where"              : 0,
}

decision = router.route(sample_q, sample_feats)
print("── Production Routing Result ──")
print(f"Question         : {sample_q}")
print(f"Routing Decision : {decision['target_pipeline']}")
print(f"Meta-Confidence  : {decision['confidence']*100:.2f}%")
print(f"DeBERTa Prob     : {decision['deberta_prob']:.4f}")
print(f"ModernBERT Prob  : {decision['modernbert_prob']:.4f}")
print(f"Encoder Disagree : {decision['disagreement']:.4f}")